In [13]:
# datasets
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# control utils
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.visualization_utils import _init_rerun
from lerobot.datasets.utils import hw_to_dataset_features
from lerobot.record import record_loop
from lerobot.utils.utils import log_say

# robot configs
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# utils
from pprint import pprint
from dotenv import load_dotenv
import os

# my code
from scripts.utils.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

ImportError: cannot import name 'HF_NAME' from 'scripts.utils.paths' (g:\My Drive\Github\IDC_Project-Sim_Real_Sim\scripts\utils\paths.py)

Recording Parameters:

In [3]:
NUM_EPISODES = 5
FPS = 30
EPISODE_TIME_SEC = 60 # how long each episode
RESET_TIME_SEC   = 10 # env reset time
TASK_DESCRIPTION = "My task description"
REPO_NAME = 'so101_table_leg_assembly'
PUSH_TO_HUB = False

Login to HF if pushing:

In [ ]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

Robot Config:

In [5]:
camera_config = {
    "wrist_cam": OpenCVCameraConfig(index_or_path=1, width=640, height=480, fps=30)
}

robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    cameras = camera_config,
    calibration_dir = CALIBS_DIR
)

teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)

robot = SO101Follower(robot_config)
teleop = SO101Leader(teleop_config)

Dataset build:

In [6]:
action_features = hw_to_dataset_features(robot.action_features, "action")
obs_features = hw_to_dataset_features(robot.observation_features, "observation")
dataset_features = {**action_features, **obs_features}
pprint(dataset_features)

{'action': {'dtype': 'float32',
            'names': ['shoulder_pan.pos',
                      'shoulder_lift.pos',
                      'elbow_flex.pos',
                      'wrist_flex.pos',
                      'wrist_roll.pos',
                      'gripper.pos'],
            'shape': (6,)},
 'observation.images.wrist_cam': {'dtype': 'video',
                                  'names': ['height', 'width', 'channels'],
                                  'shape': (480, 640, 3)},
 'observation.state': {'dtype': 'float32',
                       'names': ['shoulder_pan.pos',
                                 'shoulder_lift.pos',
                                 'elbow_flex.pos',
                                 'wrist_flex.pos',
                                 'wrist_roll.pos',
                                 'gripper.pos'],
                       'shape': (6,)}}


Build Dataset:

In [8]:
dataset = LeRobotDataset.create(
    repo_id=f"{HF_NAME}/{REPO_NAME}",
    root=f"{DATASETS_DIR}/{REPO_NAME}",
    fps=FPS,
    features=dataset_features,
    robot_type=robot.name,
    use_videos=True,
    image_writer_threads=4,
)

Recording prep:

In [9]:
_, events = init_keyboard_listener()
_init_rerun(session_name="recording")

# connect to robots
robot.connect()
teleop.connect()

ConnectionError: 
Could not connect on port 'COM7'. Make sure you are using the correct port.
Try running `python -m lerobot.find_port`


Recording Loop:

In [ ]:
episode_idx = 0
while episode_idx < NUM_EPISODES and not events["stop_recording"]:
    log_say(f"Recording episode {episode_idx + 1} of {NUM_EPISODES}")

    record_loop(
        robot=robot,
        events=events,
        fps=FPS,
        teleop=teleop,
        dataset=dataset,
        control_time_s=EPISODE_TIME_SEC,
        single_task=TASK_DESCRIPTION,
        display_data=True,
    )

    # Reset the environment if not stopping or re-recording
    if not events["stop_recording"] and (episode_idx < NUM_EPISODES - 1 or events["rerecord_episode"]):
        log_say("Reset the environment")
        record_loop(
            robot=robot,
            events=events,
            fps=FPS,
            teleop=teleop,
            control_time_s=RESET_TIME_SEC,
            single_task=TASK_DESCRIPTION,
            display_data=True,
        )

    if events["rerecord_episode"]:
        log_say("Re-recording episode")
        events["rerecord_episode"] = False
        events["exit_early"] = False
        dataset.clear_episode_buffer()
        continue

    dataset.save_episode()
    episode_idx += 1

: 

In [ ]:
log_say("Stop recording")
robot.disconnect()
teleop.disconnect()
if PUSH_TO_HUB:
    dataset.push_to_hub()

: 